# PLN Analytics Platform — Monthly ETL

Run this notebook every time new Excel exports are available. It:
1. Pulls the platform repo (which contains the `etl/` package)
2. Lets you upload the raw `.xlsx` files for the new period
3. Runs the full ETL pipeline (discover → merge same-month files → validate → clean → transform → write partitioned datasets)
4. Pushes `data/processed/` back to GitHub so the dashboard picks it up automatically

**No code changes are ever needed here for a new month** — just upload the new files and re-run.

In [ ]:
# STEP 0 — Install dependencies not preinstalled in the Colab runtime
!pip install -q pyarrow

In [ ]:
# STEP 1 — Get the repo. Replace with your actual repo URL.
# If the repo is private, use a fine-grained GitHub token instead of a plain HTTPS URL:
#   !git clone https://<TOKEN>@github.com/<org>/pln-analytics-platform.git
REPO_URL = "https://github.com/<org>/pln-analytics-platform.git"

import os
if not os.path.exists("pln-analytics-platform"):
    !git clone $REPO_URL
%cd pln-analytics-platform
!git pull

In [ ]:
# STEP 2 — Upload this period's raw Excel files into data/raw/
# (You can upload as many files as needed — e.g. all 3 ANEV slices for the
# month, plus the DLPD Pascabayar/Prabayar/Pengecekan exports — in one go.)
from google.colab import files
import shutil, os

os.makedirs("data/raw", exist_ok=True)
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, os.path.join("data/raw", filename))

print("Files now in data/raw/:")
for f in sorted(os.listdir("data/raw")):
    print(" -", f)

In [ ]:
# STEP 3 — Run the ETL pipeline
!python -m etl.run_etl --input-dir data/raw --output-dir data/processed

In [ ]:
# STEP 4 — Sanity-check the output before pushing
import pandas as pd, glob

for dataset in ["executive_kpis", "dlpd_customer", "suspect_main", "suspect_summary", "suspect_detail", "pengecekan"]:
    files_found = glob.glob(f"data/processed/{dataset}/**/*.parquet", recursive=True) or \
                  glob.glob(f"data/processed/{dataset}/**/*.csv", recursive=True)
    total_rows = 0
    for fp in files_found:
        total_rows += (pd.read_parquet(fp) if fp.endswith(".parquet") else pd.read_csv(fp)).shape[0]
    print(f"{dataset:20s} {len(files_found)} partition(s), {total_rows} total row(s)")

In [ ]:
# STEP 5 — Commit and push processed datasets so the dashboard sees them.
# Configure git identity once per Colab session:
!git config user.email "etl-bot@pln-analytics.local"
!git config user.name "PLN ETL Bot"

!git add data/processed
!git commit -m "ETL: update processed datasets"
!git push